In [ ]:
import numpy as np
import pandas as pd
from scipy.special import softmax
import csv
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig
import matplotlib.pyplot as plt

MODEL = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)


file_name = 'data.csv'
df = pd.read_csv(file_name)
eng_comments = df.loc[df.Key == 'en', 'Value'].tolist()

def save_dict_to_csv(filename, dictionary):
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['Key', 'Value'])
        for key, value in dictionary.items():
            for v in value:
                writer.writerow([key, v])

def sent_analysis(sample=list, print_res=False):
    result_output = []
    for comment in sample:
        encoded_input = tokenizer.encode_plus(comment, return_tensors='pt')
        output = model(**encoded_input)
        scores = output[0][0].detach().numpy()
        scores = softmax(scores)
        result_output.append(scores)
    if print_res:
        for index in range(len(result_output)):
            print(f'\n#{index+1} comment: "{sample[index]}"')
            for i in range(3):
                print(f'{config.id2label[i]}: {np.round(float(result_output[index][i]), 2)}')
    return result_output


def comment_by_emotional_color(scores, sample=list, save=False):
    res_dict = {}
    for value in config.id2label.values():
        res_dict[value] = []
    for index, score in enumerate(scores):
        if score[0] == max(score):
            res_dict['negative'].append(sample[index])
        elif score[1] == max(score):
            res_dict['neutral'].append(sample[index])
        elif score[2] == max(score):
            res_dict['positive'].append(sample[index])

    if save:
        save_dict_to_csv('emotional.csv', res_dict)
    return res_dict

def visualisation():
    df2 = pd.read_csv('emotional.csv')
    keys = df2['Key'].unique()
    value_len = df2['Key'].value_counts().values
    fig = plt.figure(figsize=(6, 4))
    ax = fig.add_subplot()
    ax.bar(keys, value_len)
    ax.grid()
    plt.show()


sent_analysis(eng_comments, True)